# Cross-sectional forecast reconciliation with FoReco

This notebook applies cross-sectional forecast reconciliation to hierarchical Zillow forecasts using the `FoReco` R package.

The hierarchy is:

`Country → State → Region`

The input data are stored in long format with the following schema:

`unique_id`, `ds`, `cutoff`, `y`, `forecast`, `model`, `type`, `hierarchy_level`

For each `model × cutoff` pair:

- `in_sample` forecasts are used to compute residuals.
- `out_sample` forecasts are reconciled.

In [1]:
library(FoReco)
library(arrow)
library(data.table)

cfg <- list(input_path = "Data/zillow_forecast_all_clean.parquet", 
            output_path = "Data/zillow_foreco_cross_sectional_base_and_reconciled.parquet", 
            id_sep_regex = "\\|", id_sep_out = "|", top_level = 0, middle_level = 1, bottom_level = 2)

required_cols <- c("unique_id", "ds", "cutoff", "y", "forecast", "model", "type", "hierarchy_level")

forecast_all <- as.data.table(read_parquet(cfg$input_path))
stopifnot(all(required_cols %in% names(forecast_all)))

forecast_all <- forecast_all[, ..required_cols]
forecast_all[, unique_id := as.character(unique_id)]
forecast_all[, model := as.character(model)]
forecast_all[, type := as.character(type)]
forecast_all[, ds := as.Date(ds)]
forecast_all[, cutoff := as.Date(cutoff)]
forecast_all[, y := as.numeric(y)]
forecast_all[, forecast := as.numeric(forecast)]
forecast_all[, hierarchy_level := as.integer(hierarchy_level)]

forecast_all[1:10]

Cargando paquete requerido: Matrix

Warning message:
“package ‘Matrix’ was built under R version 4.5.3”
Warning message:
“package ‘arrow’ was built under R version 4.5.3”

Adjuntando el paquete: ‘arrow’


The following object is masked from ‘package:utils’:

    timestamp




unique_id,ds,cutoff,y,forecast,model,type,hierarchy_level
<chr>,<date>,<date>,<dbl>,<dbl>,<chr>,<chr>,<int>
US,2008-10-31,2021-03-31,224158,223944.8,AutoARIMA,in_sample,0
US,2008-11-30,2021-03-31,166173,166019.0,AutoARIMA,in_sample,0
US,2008-12-31,2021-03-31,191299,191121.1,AutoARIMA,in_sample,0
US,2009-01-31,2021-03-31,145481,145350.1,AutoARIMA,in_sample,0
US,2009-02-28,2021-03-31,157695,175440.0,AutoARIMA,in_sample,0
US,2009-03-31,2021-03-31,200567,196702.4,AutoARIMA,in_sample,0
US,2009-04-30,2021-03-31,212809,224815.4,AutoARIMA,in_sample,0
US,2009-05-31,2021-03-31,231116,241035.2,AutoARIMA,in_sample,0
US,2009-06-30,2021-03-31,274155,256967.1,AutoARIMA,in_sample,0


## Initial data checks

The dataset is inspected by forecast type, hierarchy level, model, and cutoff.

Expected values in `type`:

- `in_sample`: fitted values used to compute residuals.
- `out_sample`: forecasts to be reconciled.

In [2]:
forecast_all[, .N, by = type][order(type)]
forecast_all[, .N, by = hierarchy_level][order(hierarchy_level)]
forecast_all[, .N, by = model][order(model)]
forecast_all[type == "out_sample", .N, by = cutoff][order(cutoff)]

type,N
<chr>,<int>
in_sample,3789800
out_sample,273000


hierarchy_level,N
<int>,<int>
0,11608
1,568792
2,3482400


model,N
<chr>,<int>
AutoARIMA,322700
AutoARIMAX,322700
CatBoost,282800
GRU,319550
HoltWinters,322700
KAN,319550
LightGBM,282800
MFLES,322700
NBEATSx,319550


cutoff,N
<date>,<int>
2021-03-31,54600
2022-03-31,54600
2023-03-31,54600
2024-03-31,54600
2025-03-31,54600


## Hierarchy construction

FoReco requires an aggregation matrix to represent the cross-sectional hierarchy.

For this project, the hierarchy is:

`Country → State → Region`

The `unique_id` column encodes the hierarchy as:

`Country|State|Region`

For example:

`US|CA|42387`

The aggregation matrix maps bottom-level regions to upper-level country and state totals.

In [3]:
build_hierarchy <- function(dt) {
  hierarchy <- unique(dt[, .(unique_id, hierarchy_level)])
  hierarchy[order(hierarchy_level, unique_id)]
}

get_ancestor_ids <- function(id, max_parent_level, sep_regex, sep_out) {
  parts <- strsplit(id, sep_regex)[[1]]
  max_k <- min(length(parts) - 1, max_parent_level + 1)
  if (max_k < 1) return(character(0))
  vapply(seq_len(max_k), function(k) paste(parts[seq_len(k)], collapse = sep_out), character(1))
}

get_parent_id <- function(id, parent_level, sep_regex, sep_out) {
  parts <- strsplit(id, sep_regex)[[1]]
  paste(parts[seq_len(parent_level + 1)], collapse = sep_out)
}

build_agg_mat <- function(hierarchy, cfg) {
  upper_ids <- hierarchy[hierarchy_level < cfg$bottom_level, unique_id]
  bottom_ids <- hierarchy[hierarchy_level == cfg$bottom_level, unique_id]
  agg_mat <- matrix(0, nrow = length(upper_ids), ncol = length(bottom_ids), dimnames = list(upper_ids, bottom_ids))
  for (bottom_id in bottom_ids) {
    ancestors <- get_ancestor_ids(bottom_id, cfg$bottom_level - 1, cfg$id_sep_regex, cfg$id_sep_out)
    ancestors <- intersect(ancestors, upper_ids)
    if (length(ancestors) == 0) stop("No valid ancestors found for bottom series: ", bottom_id)
    agg_mat[ancestors, bottom_id] <- 1
  }
  agg_mat
}

make_specs <- function(dt, cfg) {
  hierarchy <- build_hierarchy(dt)
  agg_mat <- build_agg_mat(hierarchy, cfg)
  top_ids <- hierarchy[hierarchy_level == cfg$top_level, unique_id]
  middle_ids <- hierarchy[hierarchy_level == cfg$middle_level, unique_id]
  bottom_ids <- hierarchy[hierarchy_level == cfg$bottom_level, unique_id]
  all_ids <- c(rownames(agg_mat), colnames(agg_mat))
  middle_rows <- match(middle_ids, rownames(agg_mat))
  list(hierarchy = hierarchy, agg_mat = agg_mat, top_ids = top_ids, middle_ids = middle_ids, bottom_ids = bottom_ids, all_ids = all_ids, middle_rows = middle_rows)
}

specs <- make_specs(forecast_all, cfg)

specs$hierarchy[, .N, by = hierarchy_level][order(hierarchy_level)]
dim(specs$agg_mat)
specs$agg_mat[1:5, 1:5]

hierarchy_level,N
<int>,<int>
0,1
1,49
2,300


[1]  50 300

,US|AK|394327,US|AL|394351,US|AL|394388,US|AL|394519,US|AL|394537
US,1,1,1,1,1
US|AK,1,0,0,0,0
US|AL,0,1,1,1,1
US|AR,0,0,0,0,0
US|AZ,0,0,0,0,0


## Matrix utilities

FoReco works with matrices, while the original forecasts are stored in long format.

The helper functions below convert long-format data into matrices with a fixed and consistent column order:

`upper-level series first, bottom-level series second`

This order is required because the aggregation matrix is defined as:

`upper-level series × bottom-level series`

In [4]:
wide_matrix <- function(dt, value_col, ids) {
  x <- dcast(dt, ds ~ unique_id, value.var = value_col, fun.aggregate = mean)
  missing_ids <- setdiff(ids, names(x))
  if (length(missing_ids) > 0) x[, (missing_ids) := NA_real_]
  setcolorder(x, c("ds", ids))
  mat <- as.matrix(x[, ..ids])
  storage.mode(mat) <- "double"
  colnames(mat) <- ids
  list(ds = x$ds, mat = mat)
}

validate_matrix <- function(mat, name) {
  if (anyNA(mat)) stop(name, " contains NA values.")
  if (any(!is.finite(mat))) stop(name, " contains non-finite values.")
  invisible(TRUE)
}

as_reco_matrix <- function(x, ids) {
  mat <- as.matrix(x)
  if (ncol(mat) != length(ids) && nrow(mat) == length(ids)) mat <- t(mat)
  if (ncol(mat) != length(ids)) stop("Unexpected reconciled matrix dimensions.")
  colnames(mat) <- ids
  storage.mode(mat) <- "double"
  mat
}

repeat_weights <- function(w, h) {
  x <- matrix(rep(as.numeric(w), times = h), nrow = h, byrow = TRUE)
  colnames(x) <- names(w)
  x
}

clean_global_weights <- function(w) {
  w[!is.finite(w) | w < 0] <- 0
  if (sum(w) <= 0) w[] <- 1 / length(w) else w <- w / sum(w)
  w
}

clean_parent_weights <- function(w, parent_ids) {
  w[!is.finite(w) | w < 0] <- 0
  for (p in unique(parent_ids)) {
    ix <- parent_ids == p
    s <- sum(w[ix], na.rm = TRUE)
    if (!is.finite(s) || s <= 0) w[ix] <- 1 / sum(ix) else w[ix] <- w[ix] / s
  }
  w
}

## Historical proportions for top-down and middle-out reconciliation

Top-down and middle-out reconciliation require historical proportions.

Two Gross-Sohl-style proportion schemes are used:

- `gsa`: average historical proportions.
- `gsf`: proportions of historical averages.

For top-down reconciliation, the country-level forecast is distributed across bottom-level regions.

For middle-out reconciliation, state-level forecasts are distributed across their corresponding regions, and the country forecast is obtained by aggregation.

In [5]:
make_topdown_weights <- function(actual, specs, cfg, scheme) {
  bottom <- actual[, specs$bottom_ids, drop = FALSE]
  top <- actual[, specs$top_ids[1]]
  if (scheme == "gsa") {
    prop <- sweep(bottom, 1, top, "/")
    prop[!is.finite(prop)] <- NA_real_
    w <- colMeans(prop, na.rm = TRUE)
  } else {
    w <- colMeans(bottom, na.rm = TRUE) / mean(top, na.rm = TRUE)
  }
  names(w) <- specs$bottom_ids
  clean_global_weights(w)
}

make_middleout_weights <- function(actual, specs, cfg, scheme) {
  bottom <- actual[, specs$bottom_ids, drop = FALSE]
  parent_ids <- vapply(specs$bottom_ids, get_parent_id, character(1), parent_level = cfg$middle_level, sep_regex = cfg$id_sep_regex, sep_out = cfg$id_sep_out)
  if (scheme == "gsa") {
    parent_values <- sapply(parent_ids, function(p) actual[, p])
    prop <- bottom / parent_values
    prop[!is.finite(prop)] <- NA_real_
    w <- colMeans(prop, na.rm = TRUE)
  } else {
    bottom_mean <- colMeans(bottom, na.rm = TRUE)
    parent_mean <- sapply(parent_ids, function(p) mean(actual[, p], na.rm = TRUE))
    w <- bottom_mean / parent_mean
  }
  names(w) <- specs$bottom_ids
  clean_parent_weights(w, parent_ids)
}

## Reconciliation methods

This notebook evaluates point forecast reconciliation methods from the cross-sectional FoReco framework.

The selected methods are:

| Method | FoReco function | Description |
|---|---|---|
| `csbu` | `csbu()` | Bottom-up reconciliation. Bottom-level forecasts are aggregated upward. |
| `csbu_sntz` | `csbu(..., sntz = TRUE)` | Bottom-up reconciliation after setting negative bottom-level forecasts to zero. |
| `cstd_gsa` | `cstd()` | Top-down reconciliation using average historical proportions. |
| `cstd_gsf` | `cstd()` | Top-down reconciliation using proportions of historical averages. |
| `csmo_gsa` | `csmo()` | Middle-out reconciliation from the state level using average historical proportions. |
| `csmo_gsf` | `csmo()` | Middle-out reconciliation from the state level using proportions of historical averages. |
| `csrec_ols` | `csrec(comb = "ols")` | Least-squares reconciliation without covariance weighting. |
| `csrec_wls` | `csrec(comb = "wls")` | Weighted least-squares reconciliation. |
| `csrec_shr` | `csrec(comb = "shr")` | Shrinkage covariance-based reconciliation using in-sample residuals. |
| `cslcc_ols` | `cslcc(comb = "ols")` | Level conditional coherent reconciliation without covariance weighting. |
| `cslcc_wls` | `cslcc(comb = "wls")` | Level conditional coherent reconciliation with weighted least squares. |
| `cslcc_shr` | `cslcc(comb = "shr")` | Level conditional coherent reconciliation with shrinkage covariance estimation. |

In [6]:
method_registry <- data.table(
  method = c("csbu", "csbu_sntz", "cstd_gsa", "cstd_gsf", "csmo_gsa", "csmo_gsf", "csrec_ols", "csrec_wls", 
             "csrec_shr", "cslcc_ols", "cslcc_wls", "cslcc_shr"),
    
  family = c("bottom_up", "bottom_up", "top_down", "top_down", "middle_out", "middle_out", "optimal", 
             "optimal", "optimal", "lcc", "lcc", "lcc"),
    
  comb = c(NA, NA, NA, NA, NA, NA, "ols", "wls", "shr", "ols", "wls", "shr"),
  nn = c(NA, "sntz", NA, NA, NA, NA, NA, NA, NA, NA, NA, NA),
  scheme = c(NA, NA, "gsa", "gsf", "gsa", "gsf", NA, NA, NA, NA, NA, NA)
)

method_registry

method,family,comb,nn,scheme
<chr>,<chr>,<chr>,<chr>,<chr>
csbu,bottom_up,NA,NA,NA
csbu_sntz,bottom_up,NA,sntz,NA
cstd_gsa,top_down,NA,NA,gsa
cstd_gsf,top_down,NA,NA,gsf
csmo_gsa,middle_out,NA,NA,gsa
csmo_gsf,middle_out,NA,NA,gsf
csrec_ols,optimal,ols,NA,NA
csrec_wls,optimal,wls,NA,NA
csrec_shr,optimal,shr,NA,NA


## Prepare one reconciliation problem

Each reconciliation problem corresponds to one forecasting model and one cutoff.

For each `model × cutoff` pair:

- `in_sample` data are used to compute residuals and historical proportions.
- `out_sample` forecasts are converted into a base forecast matrix.
- `out_sample` actual values are retained only to preserve the final output schema.

Residuals are calculated as:

`residual = y - forecast`

These residuals are used by covariance-based methods such as `csrec_shr` and `cslcc_shr`.

The data are expected to be already cleaned and aligned. Therefore, this notebook does not impute residuals or apply partial-coverage rules.

In [7]:
prepare_problem <- function(dt, model_name, cutoff_value, specs, cfg) {
  x <- dt[model == model_name & cutoff == cutoff_value]
  ins <- copy(x[type == "in_sample"])
  outs <- copy(x[type == "out_sample"])
  if (nrow(ins) == 0) stop("No in-sample data found.")
  if (nrow(outs) == 0) stop("No out-of-sample data found.")
  ins[, residual := y - forecast]
  actual_in <- wide_matrix(ins, "y", specs$all_ids)
  residuals_in <- wide_matrix(ins, "residual", specs$all_ids)
  base_out <- wide_matrix(outs, "forecast", specs$all_ids)
  actual_out <- wide_matrix(outs, "y", specs$all_ids)
  validate_matrix(actual_in$mat, "In-sample actual matrix")
  validate_matrix(residuals_in$mat, "In-sample residual matrix")
  validate_matrix(base_out$mat, "Out-of-sample base forecast matrix")
  validate_matrix(actual_out$mat, "Out-of-sample actual matrix")
  h <- nrow(base_out$mat)
  weights <- list(td_gsa = repeat_weights(make_topdown_weights(actual_in$mat, specs, cfg, "gsa"), h), td_gsf = repeat_weights(make_topdown_weights(actual_in$mat, specs, cfg, "gsf"), h), mo_gsa = repeat_weights(make_middleout_weights(actual_in$mat, specs, cfg, "gsa"), h), mo_gsf = repeat_weights(make_middleout_weights(actual_in$mat, specs, cfg, "gsf"), h))
  list(base = base_out$mat, res = residuals_in$mat, y = actual_out$mat, ds = base_out$ds, weights = weights)
}

## Apply one reconciliation method

The function below receives one row from the method registry and applies the corresponding FoReco method.

The implementation is organized by reconciliation family rather than by individual method name. This makes the workflow easier to maintain and extend.

In [8]:
null_if_na <- function(x) if (length(x) == 0 || is.na(x)) NULL else x

run_foreco <- function(expr) suppressWarnings(expr)

apply_method <- function(method_row, problem, specs, cfg) {
  family <- method_row$family
  comb <- null_if_na(method_row$comb)
  nn <- null_if_na(method_row$nn)
  scheme <- null_if_na(method_row$scheme)
  base <- problem$base
  res <- problem$res
  A <- specs$agg_mat
  all_ids <- specs$all_ids
  bottom_ids <- specs$bottom_ids
  top_id <- specs$top_ids[1]
  middle_ids <- specs$middle_ids
  reco <- if (family == "bottom_up") {
    run_foreco(csbu(base = base[, bottom_ids, drop = FALSE], agg_mat = A, sntz = identical(nn, "sntz")))
  } else if (family == "top_down") {
    weights <- problem$weights[[paste0("td_", scheme)]]
    run_foreco(cstd(base = base[, top_id], agg_mat = A, weights = weights))
  } else if (family == "middle_out") {
    weights <- problem$weights[[paste0("mo_", scheme)]]
    run_foreco(csmo(base = base[, middle_ids, drop = FALSE], agg_mat = A, weights = weights, id_rows = specs$middle_rows))
  } else if (family == "optimal" && comb == "ols") {
    run_foreco(csrec(base = base, agg_mat = A, comb = "ols"))
  } else if (family == "optimal" && comb == "wls") {
    run_foreco(csrec(base = base, agg_mat = A, comb = "wls"))
  } else if (family == "optimal" && comb == "shr") {
    run_foreco(csrec(base = base, agg_mat = A, comb = "shr", res = res))
  } else if (family == "lcc" && comb == "ols") {
    run_foreco(cslcc(base = base, agg_mat = A, comb = "ols", CCC = TRUE))
  } else if (family == "lcc" && comb == "wls") {
    run_foreco(cslcc(base = base, agg_mat = A, comb = "wls", CCC = TRUE))
  } else if (family == "lcc" && comb == "shr") {
    run_foreco(cslcc(base = base, agg_mat = A, comb = "shr", res = res, CCC = TRUE))
  } else {
    stop("Unknown method family or combination.")
  }
  as_reco_matrix(reco, all_ids)
}

## Convert reconciled matrices back to long format

FoReco returns reconciled forecasts as matrices.

This function converts each reconciled matrix back to the original long-format schema:

`unique_id`, `ds`, `cutoff`, `y`, `forecast`, `model`, `type`, `hierarchy_level`

The reconciliation method is appended to the base model name. For example:

`LightGBM__csrec_shr`

In [9]:
matrix_to_long <- function(reco, problem, model_name, cutoff_value, method_name, specs) {
  fc <- as.data.table(reco)
  y <- as.data.table(problem$y)
  setnames(fc, specs$all_ids)
  setnames(y, specs$all_ids)
  fc[, ds := problem$ds]
  y[, ds := problem$ds]
  fc_long <- melt(fc, id.vars = "ds", variable.name = "unique_id", value.name = "forecast")
  y_long <- melt(y, id.vars = "ds", variable.name = "unique_id", value.name = "y")
  out <- merge(fc_long, y_long, by = c("ds", "unique_id"))
  out[, cutoff := cutoff_value]
  out[, model := paste(model_name, method_name, sep = "__")]
  out[, type := "out_sample"]
  out <- merge(out, specs$hierarchy, by = "unique_id", all.x = TRUE)
  out[, .(unique_id, ds, cutoff, y, forecast, model, type, hierarchy_level)]
}

## Reconcile one model and one cutoff

Before running the full experiment, the pipeline is tested on a single `model × cutoff` pair.

This verifies:

- hierarchy construction;
- matrix ordering;
- residual calculation;
- historical weight calculation;
- FoReco method execution;
- long-format output reconstruction.

In [10]:
empty_forecast_table <- function() data.table(unique_id = character(), ds = as.Date(character()), cutoff = as.Date(character()), y = numeric(), forecast = numeric(), model = character(), type = character(), hierarchy_level = integer())

empty_error_table <- function() data.table(model = character(), cutoff = as.Date(character()), method = character(), error = character())

bind_or_empty <- function(x, empty_fun) {
  out <- rbindlist(x, fill = TRUE)
  if (is.null(out) || nrow(out) == 0) return(empty_fun())
  out
}

reconcile_one_cutoff <- function(dt, model_name, cutoff_value, method_registry, specs, cfg) {
  problem <- prepare_problem(dt, model_name, cutoff_value, specs, cfg)
  results <- vector("list", nrow(method_registry))
  errors <- vector("list", nrow(method_registry))
  for (i in seq_len(nrow(method_registry))) {
    method_row <- method_registry[i]
    method_name <- method_row$method
    result <- tryCatch({
      reco <- apply_method(method_row, problem, specs, cfg)
      matrix_to_long(reco, problem, model_name, cutoff_value, method_name, specs)
    }, error = function(e) {
      errors[[i]] <<- data.table(model = model_name, cutoff = cutoff_value, method = method_name, error = e$message)
      NULL
    })
    results[[i]] <- result
  }
  list(forecasts = bind_or_empty(results, empty_forecast_table), errors = bind_or_empty(errors, empty_error_table))
}

test_model <- sort(unique(forecast_all$model))[1]
test_cutoff <- sort(unique(forecast_all[type == "out_sample", cutoff]))[1]

test_run <- reconcile_one_cutoff(forecast_all, test_model, test_cutoff, method_registry, specs, cfg)
test_reconciled <- test_run$forecasts
test_errors <- test_run$errors

test_reconciled[1:10]
dim(test_reconciled)
test_errors

unique_id,ds,cutoff,y,forecast,model,type,hierarchy_level
<chr>,<date>,<date>,<dbl>,<dbl>,<chr>,<chr>,<int>
US,2021-04-30,2021-03-31,431346,378815.0,AutoARIMA__csbu,out_sample,0
US,2021-05-31,2021-03-31,434753,391210.4,AutoARIMA__csbu,out_sample,0
US,2021-06-30,2021-03-31,508881,443425.9,AutoARIMA__csbu,out_sample,0
US,2021-07-31,2021-03-31,476737,460664.5,AutoARIMA__csbu,out_sample,0
US,2021-08-31,2021-03-31,468873,454224.9,AutoARIMA__csbu,out_sample,0
US,2021-09-30,2021-03-31,438350,426522.5,AutoARIMA__csbu,out_sample,0
US,2021-10-31,2021-03-31,418327,429009.4,AutoARIMA__csbu,out_sample,0
US,2021-11-30,2021-03-31,403701,383061.0,AutoARIMA__csbu,out_sample,0
US,2021-12-31,2021-03-31,406476,397604.2,AutoARIMA__csbu,out_sample,0


[1] 42000     8

model,cutoff,method,error
<chr>,<date>,<chr>,<chr>
AutoARIMA,2021-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2021-03-31,cslcc_wls,Argument `res` is NULL.


## Cross-sectional coherence check

A reconciled forecast is cross-sectionally coherent when every upper-level forecast equals the sum of its bottom-level descendants.

For this hierarchy:

- the country forecast must equal the sum of all region forecasts;
- each state forecast must equal the sum of its corresponding region forecasts.

The diagnostic below reports the maximum absolute coherence error.

In [11]:
check_coherence <- function(dt, specs) {
  upper_ids <- rownames(specs$agg_mat)
  bottom_ids <- specs$bottom_ids
  x <- dcast(dt, model + cutoff + ds ~ unique_id, value.var = "forecast", fun.aggregate = mean)
  upper <- as.matrix(x[, ..upper_ids])
  bottom <- as.matrix(x[, ..bottom_ids])
  storage.mode(upper) <- "double"
  storage.mode(bottom) <- "double"
  expected_upper <- bottom %*% t(specs$agg_mat)
  data.table(max_abs_coherence_error = max(abs(upper - expected_upper), na.rm = TRUE))
}

check_coherence(test_reconciled, specs)

max_abs_coherence_error
<dbl>
6.984919e-10


## Select valid reconciliation tasks

Some forecasting models may contain out-of-sample forecasts for a cutoff but no corresponding in-sample fitted values.

Because this notebook uses in-sample residuals, only `model × cutoff` pairs with both `in_sample` and `out_sample` data are reconciled.

Skipped tasks are logged rather than treated as method errors.

In [12]:
get_valid_tasks <- function(dt) {
  coverage <- dt[, .N, by = .(model, cutoff, type)]
  coverage <- dcast(coverage, model + cutoff ~ type, value.var = "N", fill = 0)
  if (!"in_sample" %in% names(coverage)) coverage[, in_sample := 0L]
  if (!"out_sample" %in% names(coverage)) coverage[, out_sample := 0L]
  valid <- coverage[in_sample > 0 & out_sample > 0, .(model, cutoff)]
  skipped <- coverage[out_sample > 0 & in_sample == 0, .(model, cutoff, reason = "Missing in-sample forecasts.")]
  list(valid = valid, skipped = skipped)
}

task_info <- get_valid_tasks(forecast_all)

task_info$valid[1:10]
task_info$skipped

model,cutoff
<chr>,<date>
AutoARIMA,2021-03-31
AutoARIMA,2022-03-31
AutoARIMA,2023-03-31
AutoARIMA,2024-03-31
AutoARIMA,2025-03-31
AutoARIMAX,2021-03-31
AutoARIMAX,2022-03-31
AutoARIMAX,2023-03-31
AutoARIMAX,2024-03-31


model,cutoff,reason
<chr>,<date>,<chr>


## Run the full reconciliation experiment

The full experiment applies every selected cross-sectional reconciliation method to every valid forecasting model and cutoff.

If a specific `model × cutoff × method` combination fails, the error is stored in a log and the experiment continues.

In [13]:
run_reconciliation <- function(dt, method_registry, specs, cfg) {
  task_info <- get_valid_tasks(dt)
  tasks <- task_info$valid
  skipped <- task_info$skipped
  forecast_results <- vector("list", nrow(tasks))
  error_results <- vector("list", nrow(tasks))
  message("Valid tasks: ", nrow(tasks))
  message("Skipped tasks: ", nrow(skipped))
  for (i in seq_len(nrow(tasks))) {
    model_name <- tasks$model[i]
    cutoff_value <- tasks$cutoff[i]
    message("Running ", i, "/", nrow(tasks), " | model = ", model_name, " | cutoff = ", cutoff_value)
    task_result <- tryCatch({
      reconcile_one_cutoff(dt, model_name, cutoff_value, method_registry, specs, cfg)
    }, error = function(e) {
      list(forecasts = empty_forecast_table(), errors = data.table(model = model_name, cutoff = cutoff_value, method = NA_character_, error = e$message))
    })
    forecast_results[[i]] <- task_result$forecasts
    error_results[[i]] <- task_result$errors
  }
  list(forecasts = bind_or_empty(forecast_results, empty_forecast_table), errors = bind_or_empty(error_results, empty_error_table), skipped = skipped)
}

full_run <- run_reconciliation(forecast_all, method_registry, specs, cfg)

reconciled_all <- full_run$forecasts
error_log <- full_run$errors
skipped_log <- full_run$skipped

reconciled_all <- reconciled_all[, .(unique_id, ds, cutoff, y, forecast, model, type, hierarchy_level)]

reconciled_all[1:10]
dim(reconciled_all)
error_log
skipped_log

Valid tasks: 65

Skipped tasks: 0

Running 1/65 | model = AutoARIMA | cutoff = 2021-03-31

Running 2/65 | model = AutoARIMA | cutoff = 2022-03-31

Running 3/65 | model = AutoARIMA | cutoff = 2023-03-31

Running 4/65 | model = AutoARIMA | cutoff = 2024-03-31

Running 5/65 | model = AutoARIMA | cutoff = 2025-03-31

Running 6/65 | model = AutoARIMAX | cutoff = 2021-03-31

Running 7/65 | model = AutoARIMAX | cutoff = 2022-03-31

Running 8/65 | model = AutoARIMAX | cutoff = 2023-03-31

Running 9/65 | model = AutoARIMAX | cutoff = 2024-03-31

Running 10/65 | model = AutoARIMAX | cutoff = 2025-03-31

Running 11/65 | model = CatBoost | cutoff = 2021-03-31

Running 12/65 | model = CatBoost | cutoff = 2022-03-31

Running 13/65 | model = CatBoost | cutoff = 2023-03-31

Running 14/65 | model = CatBoost | cutoff = 2024-03-31

Running 15/65 | model = CatBoost | cutoff = 2025-03-31

Running 16/65 | model = GRU | cutoff = 2021-03-31

Running 17/65 | model = GRU | cutoff = 2022-03-31

Running 18/65 | m

unique_id,ds,cutoff,y,forecast,model,type,hierarchy_level
<chr>,<date>,<date>,<dbl>,<dbl>,<chr>,<chr>,<int>
US,2021-04-30,2021-03-31,431346,378815.0,AutoARIMA__csbu,out_sample,0
US,2021-05-31,2021-03-31,434753,391210.4,AutoARIMA__csbu,out_sample,0
US,2021-06-30,2021-03-31,508881,443425.9,AutoARIMA__csbu,out_sample,0
US,2021-07-31,2021-03-31,476737,460664.5,AutoARIMA__csbu,out_sample,0
US,2021-08-31,2021-03-31,468873,454224.9,AutoARIMA__csbu,out_sample,0
US,2021-09-30,2021-03-31,438350,426522.5,AutoARIMA__csbu,out_sample,0
US,2021-10-31,2021-03-31,418327,429009.4,AutoARIMA__csbu,out_sample,0
US,2021-11-30,2021-03-31,403701,383061.0,AutoARIMA__csbu,out_sample,0
US,2021-12-31,2021-03-31,406476,397604.2,AutoARIMA__csbu,out_sample,0


[1] 2520000       8

model,cutoff,method,error
<chr>,<date>,<chr>,<chr>
AutoARIMA,2021-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2021-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2022-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2022-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2023-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2023-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2024-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2024-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2025-03-31,csrec_wls,Argument `res` is NULL.


model,cutoff,reason
<chr>,<date>,<chr>


## Validate reconciled forecasts

After running the full experiment, the reconciled forecasts are validated by checking:

1. output schema;
2. generated model-method combinations;
3. forecast range;
4. remaining negative forecasts;
5. cross-sectional coherence;
6. skipped tasks and method-level errors.

In [15]:
names(reconciled_all)

reconciled_all[, .N, by = model][order(model)][1:30]

reconciled_all[, .(min_forecast = min(forecast, na.rm = TRUE), max_forecast = max(forecast, na.rm = TRUE), n_negative = sum(forecast < 0, na.rm = TRUE))]

check_coherence(reconciled_all, specs)

if (nrow(error_log) > 0) error_log else "No method-level errors detected."
if (nrow(skipped_log) > 0) skipped_log else "No skipped model-cutoff tasks."

[1] "unique_id"       "ds"              "cutoff"          "y"              
[5] "forecast"        "model"           "type"            "hierarchy_level"

model,N
<chr>,<int>
AutoARIMAX__csbu,21000
AutoARIMAX__csbu_sntz,21000
AutoARIMAX__cslcc_ols,21000
AutoARIMAX__cslcc_shr,21000
AutoARIMAX__csmo_gsa,21000
AutoARIMAX__csmo_gsf,21000
AutoARIMAX__csrec_ols,21000
AutoARIMAX__csrec_shr,21000
AutoARIMAX__cstd_gsa,21000


min_forecast,max_forecast,n_negative
<dbl>,<dbl>,<int>
-1366.006,589942.9,2727


max_abs_coherence_error
<dbl>
2.270099e-09


model,cutoff,method,error
<chr>,<date>,<chr>,<chr>
AutoARIMA,2021-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2021-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2022-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2022-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2023-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2023-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2024-03-31,csrec_wls,Argument `res` is NULL.
AutoARIMA,2024-03-31,cslcc_wls,Argument `res` is NULL.
AutoARIMA,2025-03-31,csrec_wls,Argument `res` is NULL.


[1] "No skipped model-cutoff tasks."

## Save base and reconciled forecasts

The final output combines the original out-of-sample base forecasts and the reconciled forecasts in a single parquet file.

The base forecasts are labeled as:

`model__base`

The reconciled forecasts are labeled as:

`model__reconciliation_method`

For example:

- `LightGBM__base`
- `LightGBM__csbu`
- `LightGBM__csrec_shr`

This format keeps the original schema and makes downstream model comparison straightforward.

In [14]:
base_out <- copy(forecast_all[type == "out_sample"])
base_out[, model := paste(model, "base", sep = "__")]
base_out <- base_out[, .(unique_id, ds, cutoff, y, forecast, model, type, hierarchy_level)]

forecast_final <- rbindlist(list(base_out, reconciled_all), fill = TRUE)
forecast_final <- forecast_final[, .(unique_id, ds, cutoff, y, forecast, model, type, hierarchy_level)]

write_parquet(forecast_final, cfg$output_path)

forecast_final[1:10]
dim(forecast_final)

unique_id,ds,cutoff,y,forecast,model,type,hierarchy_level
<chr>,<date>,<date>,<dbl>,<dbl>,<chr>,<chr>,<int>
US,2021-04-30,2021-03-31,431346,365300.1,AutoARIMA__base,out_sample,0
US,2021-05-31,2021-03-31,434753,368583.9,AutoARIMA__base,out_sample,0
US,2021-06-30,2021-03-31,508881,448254.8,AutoARIMA__base,out_sample,0
US,2021-07-31,2021-03-31,476737,465181.6,AutoARIMA__base,out_sample,0
US,2021-08-31,2021-03-31,468873,460777.9,AutoARIMA__base,out_sample,0
US,2021-09-30,2021-03-31,438350,434886.6,AutoARIMA__base,out_sample,0
US,2021-10-31,2021-03-31,418327,438403.0,AutoARIMA__base,out_sample,0
US,2021-11-30,2021-03-31,403701,390354.6,AutoARIMA__base,out_sample,0
US,2021-12-31,2021-03-31,406476,401851.2,AutoARIMA__base,out_sample,0


[1] 2793000       8

In [16]:
forecast_final[, .N, by = type]
forecast_final[, .N, by = hierarchy_level][order(hierarchy_level)]
forecast_final[, .N, by = model][order(model)][1:50]
forecast_final[, .(min_ds = min(ds), max_ds = max(ds), min_cutoff = min(cutoff), max_cutoff = max(cutoff))]

type,N
<chr>,<int>
out_sample,2793000


hierarchy_level,N
<int>,<int>
0,7980
1,391020
2,2394000


model,N
<chr>,<int>
AutoARIMAX__base,21000
AutoARIMAX__csbu,21000
AutoARIMAX__csbu_sntz,21000
AutoARIMAX__cslcc_ols,21000
AutoARIMAX__cslcc_shr,21000
AutoARIMAX__csmo_gsa,21000
AutoARIMAX__csmo_gsf,21000
AutoARIMAX__csrec_ols,21000
AutoARIMAX__csrec_shr,21000


min_ds,max_ds,min_cutoff,max_cutoff
<date>,<date>,<date>,<date>
2021-04-30,2026-03-31,2021-03-31,2025-03-31
